# MELD preparation (run once on Kaggle, CPU is fine)
Downloads MELD (~10 GB), converts clips to 16 kHz mono wav (~1.5 GB),
builds the metadata csv. Results land in /kaggle/working.
Afterwards: Output tab -> New Dataset -> name it 'meld-wavs' (private).

In [ ]:
# 1. get the project code
import os
os.chdir('/kaggle/working')
!rm -rf cognivoice-component-d
!git clone -q https://github.com/Prathikesh/cognivoice-component-d.git
print('repo ready')

In [ ]:
# 2. download and extract MELD on the scratch disk (not /kaggle/working)
import subprocess
os.makedirs('/kaggle/tmp', exist_ok=True)
os.chdir('/kaggle/tmp')
# fail early with a clear message if the kernel has no internet
ping = subprocess.run(['curl', '-sI', '--max-time', '20', 'https://huggingface.co'],
                      capture_output=True)
assert ping.returncode == 0, (
    'NO INTERNET in this kernel. Fix: verify your phone number in Kaggle '
    'Settings (Settings -> Phone verification), then rerun.')
!wget -nv https://huggingface.co/datasets/declare-lab/MELD/resolve/main/MELD.Raw.tar.gz 2>&1 | tail -1
assert os.path.exists('MELD.Raw.tar.gz'), 'download failed, see wget output above'
!tar -xzf MELD.Raw.tar.gz && rm MELD.Raw.tar.gz
os.chdir('/kaggle/tmp/MELD.Raw')
# the split archives inside also need extracting
!for f in *.tar.gz; do tar -xzf "$f" && rm "$f"; done
!ls -la /kaggle/tmp/MELD.Raw

In [ ]:
# 3. convert mp4 -> wav, 8 files in parallel per split
SPLIT_DIRS = {
    'train': 'train_splits',
    'val': 'dev_splits_complete',
    'test': 'output_repeated_splits_test',
}
for split, d in SPLIT_DIRS.items():
    src = f'/kaggle/tmp/MELD.Raw/{d}'
    dst = f'/kaggle/working/meld_wav/{split}'
    os.makedirs(dst, exist_ok=True)
    print(split, '...')
    !find {src} -name '*.mp4' | xargs -P 8 -I XX sh -c 'f="XX"; b=$(basename "$f" .mp4); ffmpeg -y -loglevel error -i "$f" -ar 16000 -ac 1 '{dst}'/$b.wav'
!du -sh /kaggle/working/meld_wav

In [ ]:
# 4. build metadata (wavs already exist, so no --convert)
os.chdir('/kaggle/working/cognivoice-component-d')
!python -m src.datasets.meld --meld-root /kaggle/tmp/MELD.Raw --wav-dir /kaggle/working/meld_wav --out /kaggle/working/metadata_meld.csv

In [ ]:
# 5. make metadata paths relative to the future dataset root, then report
import pandas as pd
meta = pd.read_csv('/kaggle/working/metadata_meld.csv')
meta['path'] = meta['path'].str.replace('/kaggle/working/', '', regex=False)
meta.to_csv('/kaggle/working/metadata_meld.csv', index=False)
print(len(meta), 'clips ready')
print(meta.groupby(['split', 'emotion']).size())
# keep the output clean: only wavs + metadata stay in /kaggle/working
!rm -rf /kaggle/working/cognivoice-component-d